# Week 1 baselines
Scores Qwen2.5-0.5B-Instruct (GPU 0) and Qwen2.5-1.5B-Instruct (GPU 1) in parallel on GSM8K test, MATH-500 and SVAMP with vLLM, greedy decoding, fp16 (T4 has no fast bf16). Results land in /kaggle/working/results.

In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm "math-verify[antlr4_13_2]" 2>&1 | tail -3
# vLLM brings its own torch build; Kaggle's preinstalled torchaudio no longer matches it
# and transformers imports it on startup, so remove it (no audio needed here).
!pip uninstall -y -q torchaudio
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__)"

In [ ]:
!python -m pytest -q tests/ 2>&1 | tail -3

In [ ]:
OUT = "/kaggle/working"
jobs = {"base-0.5b": ("Qwen/Qwen2.5-0.5B-Instruct", "0"), "base-1.5b": ("Qwen/Qwen2.5-1.5B-Instruct", "1")}
env = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
procs = {}
for name, (model, gpu) in jobs.items():
    cmd = ["python", "-m", "eval.harness", "--model", model, "--dtype", "float16", "--out", f"{OUT}/results/{name}"]
    procs[name] = subprocess.Popen(cmd, env={**env, "CUDA_VISIBLE_DEVICES": gpu},
                                   stdout=open(f"{OUT}/{name}.log", "w"), stderr=subprocess.STDOUT)
for name, p in procs.items():
    print(name, "exit code", p.wait())
for name in jobs:
    print(f"\n===== {name} =====")
    print("".join(l for l in open(f"{OUT}/{name}.log") if l.startswith("[") or "Error" in l)[-3000:])

In [ ]:
import json
rows = []
for name in jobs:
    path = f"{OUT}/results/{name}/summary.json"
    if not os.path.exists(path):
        print("missing", path); continue
    for ds, s in json.load(open(path))["datasets"].items():
        rows.append(f"{name:10s} {ds:8s} acc {s['accuracy']*100:5.1f}%  boxed {s['boxed_rate']*100:5.1f}%  "
                    f"trunc {s['truncated_rate']*100:4.1f}%  tokens {s['avg_tokens']:4.0f}  {s['seconds']}s")
print("\n".join(rows))
os.makedirs(f"{OUT}/failures", exist_ok=True)
for name in jobs:
    for ds in ("gsm8k", "math500", "svamp"):
        src = f"{OUT}/results/{name}/{ds}.jsonl"
        if os.path.exists(src):
            subprocess.run(["python", "-m", "eval.failures", src, "-n", "20"],
                           stdout=open(f"{OUT}/failures/{name}-{ds}.md", "w"), check=True)